# Rock Game Prototype Checkpoint Walkthrough

This notebook runs the split-module prototype flow using `GameMaster`.

Checkpoints covered:
- create a playable game
- inspect starter rocks and market pods
- queue parents and advance a generation
- buy potions
- buy a defined-trait rock
- buy a market pod and keep a child
- serialize and reload the game
- render one rock, rock grids, and lineage trees
- build a larger multi-generation tree smoke check

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Develepor_X":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import Rock_Genetics.rock_genetic_helper as genetics
from Rock_GameState.rock_game_state_helper import GameMaster
from Rock_Serialization.rock_serialization_helper import game_to_json_string, game_from_json_string
from Rock_Drawing.rock_draw_machine import draw_rock
from Rock_Drawing.rock_lineage_drawing_helper import TreeDrawer, show_rocks as show_rock_grid
import matplotlib.pyplot as plt

print(f"Loaded project from: {PROJECT_ROOT}")

## Helper Display Functions

In [ ]:
def show_status(game, label="Game status"):
    print(f"\n=== {label} ===")
    for key, value in game.update_display().items():
        if key == "events":
            continue
        print(f"{key}: {value}")
    print("recent events:")
    for event in game.events[-6:]:
        print(f"- {event}")


def list_rocks(game, label="Rocks"):
    print(f"\n=== {label} ===")
    for line in game.show_rocks():
        print(line)


def show_market(game, label="Market pods"):
    print(f"\n=== {label} ===")
    for offer in game.market_pods:
        used = "used" if offer.used else "available"
        print(f"{offer.offer_id}: {offer.name} | ${offer.price} | {used} | {offer.tagline}")

## Checkpoint 1: Create A Playable Game

In [ ]:
game = GameMaster(seed=101, starting_money=40)

show_status(game, "New GameMaster")
list_rocks(game, "Starter rocks")
show_market(game)

## Checkpoint 2: Queue Parents And Advance Generation

In [ ]:
males = [rock for rock in game.rocks.values() if rock.sex == genetics.Sex.MALE and rock.status == genetics.RockStatus.ACTIVE]
females = [rock for rock in game.rocks.values() if rock.sex == genetics.Sex.FEMALE and rock.status == genetics.RockStatus.ACTIVE]

parent_a = males[0]
parent_b = females[0]

queued = game.add_pair_to_queue(parent_a.id, parent_b.id)
print(f"Queued pair: #{queued.parent_a_id} x #{queued.parent_b_id}")

children = game.advance_generation()
print(f"Children created: {len(children)}")
for child in children:
    print(f"- child #{child.id}, parents={child.parent_ids}, gen={child.generation}, status={child.status.value}, value=${child.value}")

show_status(game, "After breeding generation")
list_rocks(game, "Rocks after breeding")

## Checkpoint 3: Buy Potions

In [ ]:
for potion_key in ["fertility", "reroll", "mutation", "anti_craisen"]:
    try:
        game.buy_potion(potion_key)
        print(f"Bought potion: {potion_key}")
    except ValueError as exc:
        print(f"Could not buy {potion_key}: {exc}")

show_status(game, "After potion shopping")

## Checkpoint 4: Buy A Defined-Trait Rock

In [ ]:
defined_rock = game.buy_defined_trait_rock(
    {
        "gender": "female",
        "color": "34",
        "eyes": "11",
        "mouths": "22",
    },
    random_fill=False,
)

print(f"Bought defined rock #{defined_rock.id}: {defined_rock.name.full_name}")
print(f"sex: {defined_rock.sex.value}")
print(f"color phenotype: {defined_rock.genotype.genes['color'].phenotype}")
print(f"eyes phenotype: {defined_rock.genotype.genes['eyes'].phenotype}")
print(f"mouth phenotype: {defined_rock.genotype.genes['mouths'].phenotype}")

show_status(game, "After defined-trait purchase")

## Checkpoint 5: Buy A Market Pod And Keep One Child

In [ ]:
show_market(game, "Available market pods before buying")

available_offer = next(offer for offer in game.market_pods if not offer.used)
pending = game.market_manager.buy_market_pod(game, available_offer.offer_id)

print(f"Bought pod: {pending.offer.name}")
print(f"candidate children: {len(pending.children)}")
for index, child in enumerate(pending.children):
    print(f"{index}: temp child #{child.id}, sex={child.sex.value}, value=${child.value}")

kept_child = game.market_manager.choose_market_pod_child(game, 0)
print(f"Kept market child #{kept_child.id}, parents={kept_child.parent_ids}")

show_status(game, "After market pod child selection")
list_rocks(game, "Rocks after market pod")

## Checkpoint 6: Serialize And Reload

In [ ]:
save_json = game_to_json_string(game)
loaded_game = game_from_json_string(save_json)

print(f"save length: {len(save_json)} characters")
print(f"original rocks: {len(game.rocks)}")
print(f"loaded rocks: {len(loaded_game.rocks)}")
print(f"original money: {game.money}")
print(f"loaded money: {loaded_game.money}")
print(f"loaded generation: {loaded_game.generation}")

show_status(loaded_game, "Loaded game status")

## Checkpoint 7: Draw One Rock, A Rock Set, And The Full Tree

In [ ]:
one_rock = next(iter(game.rocks.values()))

fig, ax = plt.subplots(figsize=(4, 4))
draw_rock(one_rock, ax=ax)
plt.show()

show_rock_grid(game.rocks, cols=4, title="Current Rock Set", sort_by_generation=True)
plt.show()

tree_fig = TreeDrawer(game=game, canvas_width=1200, canvas_height=800).draw()
tree_fig.show(config={"scrollZoom": True, "displayModeBar": True})

## Checkpoint 7B: Larger Multi-Generation Tree


In [ ]:
large_game = GameMaster(seed=303, starting_money=120)

for step in range(3):
    queued = 0
    used_ids = set()
    active_males = [
        rock for rock in large_game.rocks.values()
        if rock.sex == genetics.Sex.MALE and rock.status == genetics.RockStatus.ACTIVE
    ]
    active_females = [
        rock for rock in large_game.rocks.values()
        if rock.sex == genetics.Sex.FEMALE and rock.status == genetics.RockStatus.ACTIVE
    ]

    for male in active_males:
        if queued >= large_game.max_pairs_per_generation or male.id in used_ids:
            continue
        for female in active_females:
            if female.id in used_ids:
                continue
            try:
                large_game.add_pair_to_queue(male.id, female.id)
            except ValueError:
                continue
            used_ids.update({male.id, female.id})
            queued += 1
            break

    if queued == 0:
        print(f"Generation step {step + 1}: no valid pairs left.")
        break

    new_children = large_game.advance_generation()
    print(
        f"Generation {large_game.generation}: queued {queued} pair(s), "
        f"created {len(new_children)} child rock(s), total rocks={len(large_game.rocks)}"
    )

list_rocks(large_game, "Large game rocks")
show_rock_grid(
    large_game.rocks,
    cols=6,
    title="Large Multi-Generation Rock Set",
    sort_by_generation=True,
)
plt.show()

large_tree_fig = TreeDrawer(game=large_game, canvas_width=1500, canvas_height=950).draw()
large_tree_fig.show(config={"scrollZoom": True, "displayModeBar": True})


## Checkpoint 8: Relationship Validation Smoke Check

In [ ]:
first_child = children[0]
related_parent = game.get_rock(first_child.parent_ids[0])

try:
    game.add_pair_to_queue(related_parent.id, first_child.id)
except ValueError as exc:
    print("Relationship validation blocked breeding as expected:")
    print(exc)